## [Predicting Electric Vehicle Purchases](https://www.kaggle.com/competitions/playground-series-s6e9/code?competitionId=125219&sortBy=scoreDescending&excludeNonAccessedDatasources=true)


Playground Series - Season 6 Episode 9

| |  | | public | &nbsp; | | &nbsp; | &nbsp; | &nbsp; |
|:-|:-:|:-:|:-:| :-: | :-: | :-: | :-: | :-: |
| **1**. |||[0.94_614](https://www.kaggle.com/code/sumitsarkar969/catboost-playground-series-s6e9?scriptVersionId=349123971)|&nbsp;v.8&nbsp;| [Catboost . Playground Series - S6E9](https://www.kaggle.com/code/sumitsarkar969/catboost-playground-series-s6e9) | contributor | [Sumit Sarkar](https://www.kaggle.com/sumitsarkar969) | India |
| **2**. |||[0.94_621](https://www.kaggle.com/code/yekenot/ps-s6-e9-realmlp-pytorch?scriptVersionId=348364380)|&nbsp;v.3&nbsp;| [PS.S6.E9: RealMLP · PyTorch](https://www.kaggle.com/code/yekenot/ps-s6-e9-realmlp-pytorch/notebook) | master  | [Vladimir Demidov](https://www.kaggle.com/yekenot) | Russia |
| **3**. |||[0.94_634](https://www.kaggle.com/code/ayodejiibrahimlateef/ev-s6e9-lightgbm-cv5-lb-0-94634?scriptVersionId=349277144)|&nbsp;v.1&nbsp;| [EV S6E9 LightGBM CV5 . LB 0.94634](https://www.kaggle.com/code/ayodejiibrahimlateef/ev-s6e9-lightgbm-cv5-lb-0-94634) | expert | [Ayodeji](https://www.kaggle.com/ayodejiibrahimlateef) | Denmark |
| **4**. |||[0.94_639](https://www.kaggle.com/code/najiama/xgboost-triple-te-dynamic-pruning-lb-0-94639?scriptVersionId=349376806)|&nbsp;v.1&nbsp;| [XGBoost . Triple-TE & Dynamic Pruning (LB 0.94639)](https://www.kaggle.com/code/najiama/xgboost-triple-te-dynamic-pruning-lb-0-94639?scriptVersionId=349376806) | expert  | [Naji](https://www.kaggle.com/najiama) | Brazil |
||||| | [ **participiants** ] . [ **weights** ] |asc/desc|**correct weight**||
|| | | [0.94_640](https://www.kaggle.com/code/nina2025/ps-s6e9-h-blend-5?scriptVersionId=349424659) | &nbsp; &nbsp; [**v**.**1**](#Version.1,2) &nbsp; &nbsp;  |[ **1**. **2**. **3**. **4**.] . [ 1/4 + 1/4 + 1/4 + 1/4 ] | 30 x 70 |[ +0.08, -0.01,-0.03,-0.05 ]|
|| | | [0.94_643](https://www.kaggle.com/code/nina2025/ps-s6e9-h-blend-5?scriptVersionId=349426254) | &nbsp; &nbsp; [**v**.**2**](#Version.1,2) &nbsp; &nbsp;  |[ **1**. **2**. **3**. **4**.] . [ 0.11+0.25+0.25+0.39 ] | 30 x 70 |[ +0.074, -0.01,-0.024,-0.04 ]|
|| | | [?]() | &nbsp; &nbsp; [**v**.**3**](#Version.3) &nbsp; &nbsp;  |[ v.2 + Round Robin ] . [ 0.10 + 0.90 ] | rank |[ - ]|

In [ ]:
import numpy as np, pandas as pd
import os,ast,copy, polars as pl, shutil, matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings('ignore')

# An example of working with Seaborn is taken from a notebook:
# https://www.kaggle.com/code/likithagedipudi/kaggle-success-factors
# Presented by an expert from Buffalo, New York, United States - Likitha Gedipudi
# https://www.kaggle.com/likithagedipudi

from bokeh.plotting import figure, gridplot 
from bokeh.io import output_file, show, output_notebook
output_notebook()

def seaborn_display_1(params, df_cross):
    
    colors = [subm['color'] for subm in params['subm']]
    
    def dossier(js,subms,cols):
        def quant(i,js,subms,cols):
            return {"c" : i, "q" : sum([1 for subm in cols[i] if subm == subms[js]])}
        return {
            'name' : subms[js],
            'q_in' : [quant(i,js,subms,cols) for i in range(len(subms))]
        }
    alls = pd.read_csv(f'tida_desc.csv')
    matrix = [ast.literal_eval(str(row.alls)) for row in alls.itertuples()]
    subms = sorted(matrix[0])
    cols = [[data[i] for data in matrix] for i in range(len(subms))]
    df_subms = pd.DataFrame({f'col_{i}': [x[i] for x in matrix] for i in range(len(subms))})
    dossiers = [dossier(js,subms,cols) for js in range(len(subms))]
    subm_names = [one_dossier['name'] for one_dossier in dossiers]
    
    nqs,qss,i = [],[],0
    
    for one_dossier in dossiers: 
        qs = [one['q'] for one in one_dossier['q_in']]
        x_names = [name.replace("Group","").replace("subm_","") for name in subm_names]
        qss.append(qs)
        nqs.append({'n':x_names, 'q':qs})
        i+=1
    
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams['font.size']      = 7
    plt.rcParams['axes.titlesize'] = 9
    plt.rcParams['axes.labelsize'] = 8

    len_nqs = len(nqs) if len(nqs) > 3 else 4

    fig, axes = plt.subplots(2, len_nqs, figsize=(2.1*len_nqs, 4))
    
    def ric(j, i, n, q, colors):
        ax1 = axes[j, i]
        bars1 = ax1.bar(n, q, color=colors)
        ax1.set_title(f'ratios in alls.col {i}', fontweight='bold')
        ax1.set_xticklabels(n, rotation=45, ha='right')
        i = 0
        for bar, val in zip(bars1, q):
            if i != 0:
                ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, 
                         f'{val:,.0f}', ha='center', va='bottom', fontsize=8)
            i += 1

    lm_max = 0
    for i in range(len(nqs)):
        for q in nqs[i]['q']:
            if q > lm_max: lm_max = q  
    lm_colors = ['whitesmoke'] + colors
    for nq in nqs:
        nq['n'].insert(0,'')
        nq['q'].insert(0, lm_max*1.13)

    for i in range(len(nqs)): ric(0, i, nqs[i]['n'], nqs[i]['q'], lm_colors)

    ax10 = axes[1, 0]
    #--------------------------------------------------------------------------------
    sub_wts = params['subwts']
    main_wts = [subm['weight'] for subm in params['subm']]
    mms,acc_mass = [],[]
    for j in range(len(dossiers)):
        one_dossier = dossiers[j]
        qs = [one['q'] for one in one_dossier['q_in']]
        mm = [qs[h] * (main_wts[j] + sub_wts[h]) for h in range(len(sub_wts))]
        mass = sum(mm)
        mms.append(mm)
        acc_mass.append(round(mass))
    acc_mass = acc_mass
    y_names = [str(mass) + " - " + name for name,mass in zip(subm_names,acc_mass)]
    colors  .insert(0,'whitesmoke')
    y_names .insert(0,'')
    acc_mass.insert(0, max(acc_mass) * 1.13)
    irbis = ax10.barh(y_names[::-1], acc_mass[::-1], color=colors[::-1])
    ax10.set_title('relations of g.masses', fontweight='bold')
    ax10.set_xlabel('weights')

    ax11 = axes[1, 1]
    #--------------------------------------------------------------------------------
    corr_matrix = alls[subm_names].corr()
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    mask = 0.5*mask
    sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', 
                center=0,fmt='.3f', square=True, linewidths=0.5, ax=ax11,
                cbar_kws={'shrink': 0.8})
    ax11.set_title('Correlation Matrix', fontweight='bold', fontsize=9)
    ax11.set_xticklabels(subm_names, rotation=45, ha='right')

    ax13 = axes[1,3]
    #--------------------------------------------------------------------------------
    ip1 = len(nqs)-4
    ip2 = len(nqs)-2
    sample  = alls.sample(min(100_000, len(alls)), random_state=42)
    scatter = ax13.scatter(np.log10(sample[subm_names[ip2]] + 1), 
                           np.log10(sample[subm_names[ip1]] + 1), 
                           c=sample[params['id_target'][1]], cmap='viridis', alpha=0.5,s=10)
    ax13.set_title(f'{subm_names[ip2]} vs {subm_names[ip1]}', fontweight='bold', fontsize=8)
    plt.colorbar(scatter, ax=ax13, label='Score')

    if len(nqs) >= 5:
        ax14 = axes[1,4]
        #----------------------------------------------------------------------------
        ip1 = len(nqs)-1
        ip2 = len(nqs)-2
        sample  = alls.sample(min(100_000, len(alls)), random_state=42)
        scatter = ax14.scatter(np.log10(sample[subm_names[ip2]] + 1), 
                               np.log10(sample[subm_names[ip1]] + 1), 
                               c=sample[params['id_target'][1]], cmap='viridis', alpha=0.5,s=10)
        ax14.set_title(f'{subm_names[ip2]} vs {subm_names[ip1]}', fontweight='bold', fontsize=8)
        plt.colorbar(scatter, ax=ax14, label='Score')

    ax12 = axes[1, 2]
    #--------------------------------------------------------------------------------
    colors = [subm['color'] for subm in params['subm']]
    
    for subm,color in zip(subm_names,colors):
        subm_data = alls[subm]
        ax12.hist(np.log10(subm_data + 1), bins=30, alpha=0.5, label=subm, color=color)
    ax12.set_title('(Log Scale)', fontweight='bold')
    ax12.set_xlabel('log10')
    ax12.set_ylabel('Frequency')
    ax12.legend()
 
    plt.tight_layout()
    plt.show()


def matrix_vs(path,fs_names, params):
    _id     = params['id_target'][0]
    _target = params['id_target'][1]
    def load(path,fs_names):
        dfs = [pd.read_csv(path + name_subm +'.csv') for name_subm in fs_names]
        for i in range(len(dfs)):
            dfs[i] = dfs[i].rename(columns={_target: f'{fs_names[i]}'})
        dfsm = pd.merge(dfs[0], dfs[1], on=_id)
        for i in range(2,len(dfs)):
            dfsm = pd.merge(dfsm,dfs[i],on=_id)
        return dfsm   
    def make_list_vs(fs_names):
        list = []
        for i in range(0,len(fs_names)-1):
            for j in range(i+1,len(fs_names)):
                list.append(fs_names[i] + "_vs_" + fs_names[j])
        return list
    def get_mvs(dfs, list_vs):
        def get_abs_distance(x,t1,t2):
            return abs(x[t1]-x[t2])
        for vs in list_vs:
            t = vs.split('_vs_')
            dfs[vs] = dfs.apply(lambda x: get_abs_distance(x,t[0],t[1]), axis=1)
        return dfs   
    def distance_vs(name, st_names, list_vs, dfs):
        distances = []
        for st in st_names:
            vs_between = name + "_vs_" + st
            if vs_between not in list_vs:
                distances.append(0)
            else: distances.append(round(dfs[vs_between].sum()))
        return distances
    dfs = load(path,fs_names)
    list_vs = make_list_vs(fs_names)
    mvs = get_mvs(dfs, list_vs)
    m1 = pd.DataFrame({'subm':fs_names})
    m2 = pd.DataFrame({ name :distance_vs(name, fs_names, list_vs, mvs) for name in fs_names})
    matrix = pd.concat([m1,m2],axis=1)
    return matrix


def display_distances(params):
    files = [subm['name'] for subm in params['subm']]
    distances = matrix_vs ( params['path'], files, params )            
    display(distances)


def seaborn_display_2(params, file_name_cross=""):
    display_distances(params)
    plt.figure(figsize=(9, 2.5))
    for subm in params['subm']:
        pred = pd.read_csv(params['path']+subm['name']+'.csv')[params['id_target'][1]]
        sns.kdeplot(pred, label = subm['name'], linewidth = 0.5)
    if file_name_cross != '':
        pred = pd.read_csv(file_name_cross)[params['id_target'][1]]
        sns.kdeplot(pred, label = 'blend', linewidth = 1, linestyle = 'dashed')
    plt.title("KDE")
    plt.xlabel("target")
    plt.ylabel("Density")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


def h_blend( params, _update={}, show_details=False, subm=''):    
    if _update != {}: params.update(_update)
    file_short_names = [subm['name'] for subm in params['subm']]
    _id,_target = params['id_target'][0], params['id_target'][1]
    
    def read(params,i):
        target_new_name = params["subm"][i]["name"]
        file_subm = params["path"] + target_new_name + ".csv"
        df = pl.read_csv(file_subm).rename({_target:target_new_name})
        return df 
        
    def merge(dfs_subm):
        df_subms =  dfs_subm[0].join(dfs_subm[1], on=[_id], how="inner")
        for i in range(2, len(params["subm"])): 
            df_subms = df_subms.join(dfs_subm[i], on=[_id], how="inner")
        return df_subms
        
    def da(params, show_details, sorting_direction):
        
        df_subms = merge([read(params,i) for i in range(len(params["subm"]))])
        cols = [col for col in df_subms.columns if col != _id]
        short_name_cols = [c for c in cols]
            
        def summa(x,cs,wts,ic_alls): 
            return sum([x[cs[j]] * (wts[0][j] + wts[1][ic_alls[j]]) for j in range(len(cs))])
         
        correct = [wt for wt in params["subwts"]]
        weights = [subm['weight'] for subm in params["subm"]]
        
        def f_corr(x, cs=cols, w=weights, cw=correct):
            ic = [x['alls'].index(c) for c in short_name_cols]
            cS = [x[cols[j]] * (w[j] + cw[ic[j]]) for j in range(len(cols))]
            return sum(cS)
            
        def f_alls(x, sd=sorting_direction,cs=cols):
            tes = {c: x[c] for c in cs}.items()
            return [t[0] for t in sorted(tes,key=lambda k:k[1],reverse=sd=='desc')]

        df_subms = df_subms.with_columns(
            pl.struct(cols)         .map_elements(lambda x: f_alls(x))
            .alias("alls"),
        )
        df_subms = df_subms.with_columns(
            pl.struct(cols+['alls']).map_elements(lambda x: f_corr(x), return_dtype=pl.Float32)
            .alias(sorting_direction)
        )
        df_subms = df_subms.rename({c:nc for c,nc in zip(cols, short_name_cols)})

        if sorting_direction=='desc': 
            return df_subms, params['type_sort'][2]
        if sorting_direction=='asc': 
            return df_subms, params['type_sort'][1]

    def pandas_save(polars_df, params=params, filename='tida_desc.csv'):
        pandas_df = polars_df.to_pandas()
        pandas_df['alls'] = pandas_df.apply(lambda x:str(x['alls']).replace(' ',' ,'),axis=1)
        pandas_df . to_csv(filename, index=False)
            
    def ensemble (params, show_details): 
        dfD, wtsD = da(params, show_details,'desc')
        dfA, wtsA = da(params, show_details,'asc')
        dfA = dfA[['id','asc']]
        dfj = dfD.join(dfA, on="id", how="left")
        dfe = dfj.with_columns((pl.col("desc") *wtsD + wtsA* pl.col("asc")).alias(_target))
        pandas_save(dfe, params=params)
        if show_details: 
            display(dfe.head(4))
        return dfe[[_id,_target]]
    
    df = ensemble(params, show_details)
    if subm != '': df.write_csv(subm)
    if show_details == 'seaborn':
        seaborn_display_1(params, df)
        seaborn_display_2(params, file_name_cross=subm)
    return df


def h_blend_hs(params, df_sample, target): 
    
    def read_data(subm_name):
        return pd.read_csv(params['path']+subm_name+'.csv')[params['id_target'][1]]

    subms_datas = [{'subm_name':read_data(subm['name'])} for subm in params['subm']]
    matrix      = np.vstack([subm_one_data['subm_name'] for subm_one_data in subms_datas]).T
    main_wts    = np.asarray([subm['weight'] for subm in params['subm']], dtype=float)
    correct_wts = np.asarray(params['subwts'], dtype=float)

    def side(reverse):
        order = np.argsort(-matrix if reverse else matrix, axis=1)
        out = np.zeros(matrix.shape[0], dtype=float)
        for rank_idx in range(matrix.shape[1]):
            col_idx = order[:, rank_idx]
            new_weights = main_wts[col_idx] + correct_wts[rank_idx]
            out += matrix[np.arange(matrix.shape[0]), col_idx] * new_weights
        return out

    ASC_,DESC_ = side(False),side(True)

    df_sample[target] = ASC_*params['type_sort'][1] + DESC_*params['type_sort'][2]

    return df_sample

## Version.1,2

In [ ]:
%%time

params = {
  'path'     : '/kaggle/input/datasets/nina2025/ps-s6e9-13/',
  'id_target': ['id', 'Will_Buy_EV'],
  'type_sort': ['asc/desc', 0.30, 0.70 ],
  'subwts'   : [ +0.074, -0.01, -0.024, -0.04 ],
  'subm'     : [ 
      {'name': f'0.94614', 'weight':0.11, 'color':'navy'},
      {'name': f'0.94621', 'weight':0.25, 'color':'darkorange'},
      {'name': f'0.94634', 'weight':0.25, 'color':'darkgreen'},
      {'name': f'0.94639', 'weight':0.39, 'color':'crimson'},
]}
df_hb_cross = h_blend(params, show_details='seaborn', subm='cross.csv')
print(df_hb_cross)

## Version.3

In [ ]:
cross = '/kaggle/working/cross.csv'

path = '/kaggle/input/notebooks/nina2025/ps-s6e9-round-robin/submission.csv'

round_robin = pd.read_csv(path) .set_index('id')['Will_Buy_EV']
hb_cross    = pd.read_csv(cross).set_index('id')['Will_Buy_EV'].reindex(round_robin.index)

In [ ]:
from scipy.stats import rankdata

rank = lambda s: pd.Series(rankdata(s) / len(s), index=s.index)

rank_cross, rank_robin = rank(hb_cross), rank(round_robin)

data =  rank_cross * 0.10 + 0.90 * rank_robin  # v3.LB = 0.94649
data =  rank_cross * 0.03 + 0.97 * rank_robin  # v4.LB = 0.94649 > v3
data =  rank_cross * 0.01 + 0.99 * rank_robin  # v5.LB = ?

print(hb_cross,"\n\n",round_robin,"\n\n-------\n\n",rank_cross,"\n\n",rank_robin)

In [ ]:
for file in 'cross,tida_desc'.split(','): 
    if os.path.isfile(file+'.csv'): os.remove(file+'.csv')

## Submit

In [ ]:
submission = pd.DataFrame({'id': data.index, 'Will_Buy_EV': data.to_numpy()})

submission . to_csv("submission.csv", index=False)
submission